# Neural Networks, Explained Simply
Neurons, layers, activations and training, built with PyTorch. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A factory recorded 500 readings from a machine: vibration (mm/s) and temperature (C), and whether a fault followed (1) or not (0). The boundary between them is curved.

### 500 sensor readings
- `machine_sensors.csv`: 500 readings: vibration, temperature, fault (1 = yes)

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "neural-networks/machine_sensors.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Look at the readings
- `c=df["fault"]`: red for fault, blue for no fault
- `the picture`: two interlocking curved shapes: no straight line can separate them

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["vibration"], df["temperature"],
            c=df["fault"], cmap="coolwarm", s=12)
plt.xlabel("vibration (mm/s)"); plt.ylabel("temperature (C)")
plt.show()

## 2. One neuron by hand
A neuron multiplies each input by a **weight**, adds them up with a **bias**, then applies an **activation function**. **ReLU(z) = max(0, z)**; **sigmoid(z) = 1 / (1 + e^-z)**.

### One neuron
- `torch.tensor`: PyTorch's version of a NumPy array
- `(w * x).sum() + b`: 0.5 x 2 + (-1) x 3 + 1 = -1
- `torch.relu`: max(0, z): negative becomes 0
- `torch.sigmoid`: squashes z into 0 to 1

In [ ]:
import torch

x = torch.tensor([2.0, 3.0])
w = torch.tensor([0.5, -1.0])
b = torch.tensor(1.0)
z = (w * x).sum() + b
print("weighted sum z:", z.item())
print("ReLU:          ", torch.relu(z).item())
print("sigmoid:       ", round(torch.sigmoid(z).item(), 3))

## 3. Prepare the data
Scale the inputs, split into training and test sets, and turn them into PyTorch tensors.

### Scale, split, convert
- `StandardScaler`: neural networks train much better on scaled inputs
- `dtype=torch.float32`: decimal numbers, the type PyTorch expects
- `.unsqueeze(1)`: turn 375 labels into a column: shape (375, 1)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["vibration", "temperature"]].values
y = df["fault"].values
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0)
scaler = StandardScaler().fit(X_tr)
X_tr_t = torch.tensor(scaler.transform(X_tr), dtype=torch.float32)
X_te_t = torch.tensor(scaler.transform(X_te), dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
print(X_tr_t.shape, y_tr_t.shape)

### A straight-line model first
- `LogisticRegression`: draws a straight boundary
- `the output`: decent, but it can't follow the curves

In [ ]:
from sklearn.linear_model import LogisticRegression

lin = LogisticRegression().fit(scaler.transform(X_tr), y_tr)
acc = lin.score(scaler.transform(X_te), y_te)
print("straight line, test accuracy:", round(acc, 3))

## 4. A neural network
Layers of neurons: 2 inputs, two hidden layers of 16 neurons with ReLU, and 1 output. Train by gradient descent on the log loss.

### Build the network
- `nn.Linear(2, 16)`: a layer of 16 neurons, each connected to the 2 inputs
- `nn.ReLU()`: the activation after each hidden layer
- `nn.Linear(16, 1)`: one output neuron: a score for 'fault'
- `p.numel()`: count every weight and bias

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1))
print(model)
print("weights and biases:", sum(p.numel() for p in model.parameters()))

### Train it
- `BCEWithLogitsLoss`: log loss, with the sigmoid built in
- `Adam`: a smart version of gradient descent
- `zero_grad()`: clear the slopes from the last step
- `loss.backward()`: backpropagation: work out the slope for every weight
- `optimizer.step()`: nudge every weight downhill
- `epoch`: one pass over all the training data

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1, 501):
    optimizer.zero_grad()
    loss = loss_fn(model(X_tr_t), y_tr_t)
    loss.backward()
    optimizer.step()
    if epoch in [1, 10, 50, 100, 500]:
        print(f"epoch {epoch:>3}: loss {loss.item():.3f}")

### Test accuracy
- `torch.no_grad()`: just predicting, no training, so no slopes needed
- `torch.sigmoid`: turn scores into probabilities
- `prob >= 0.5`: predict fault above 50%

In [ ]:
with torch.no_grad():
    prob = torch.sigmoid(model(X_te_t)).squeeze().numpy()
acc = ((prob >= 0.5) == y_te).mean()
print("neural network, test accuracy:", round(acc, 3))

### Straight line vs neural network
- `np.meshgrid`: a fine grid of readings covering the chart
- `contourf(..., levels=[0, 0.5, 1])`: shade where each model predicts fault
- `the picture`: the line cuts straight across; the network bends around both shapes

In [ ]:
import numpy as np

xx, yy = np.meshgrid(np.linspace(0, 6.5, 200), np.linspace(45, 80, 200))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    g = torch.tensor(grid, dtype=torch.float32)
    nn_prob = torch.sigmoid(model(g)).numpy().ravel()
lin_prob = lin.predict_proba(grid)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, p, name in [(axes[0], lin_prob, "straight line"),
                    (axes[1], nn_prob, "neural network")]:
    ax.contourf(xx, yy, p.reshape(xx.shape), levels=[0, 0.5, 1],
                cmap="coolwarm", alpha=0.3)
    ax.scatter(df["vibration"], df["temperature"],
               c=df["fault"], cmap="coolwarm", s=8)
    ax.set_title(name)
plt.show()

## 5. Mini project: a fault alarm
Give the network new readings.

### Check new readings
- `readings`: three new pairs of readings
- `scaler.transform`: scale them exactly like the training data

In [ ]:
readings = np.array([[1.5, 72], [3.5, 55], [2.5, 64]])
r = torch.tensor(scaler.transform(readings), dtype=torch.float32)
with torch.no_grad():
    p = torch.sigmoid(model(r))
for (v, t), pr in zip(readings, p.squeeze().numpy()):
    print(f"vibration {v}, temperature {t}: {pr:.0%} fault risk")

## Try it yourself
1. Remove the ReLU layers from the network. What happens to the accuracy and the boundary?
2. Change the hidden layers from 16 to 4 neurons. Is it still good?
3. Work out the neuron by hand for inputs (1, 1) with the same weights and bias.